# Limpieza Avanzada de Datos: El Caso de Netflix

## 🎯 Objetivos
- Ejecutar una auditoría profunda de datos para identificar inconsistencias.
- Aplicar técnicas de **Normalización de Texto** (Regex, Lowercase, Stripping).
- Resolver inconsistencias de nombres utilizando **Lógica Difusa** (*Fuzzy Matching*).
- Consolidar múltiples fuentes de datos mediante la fusión (*merging*) de DataFrames.

## 1. Introducción

En el mundo real, los datos rara vez están limpios. A menudo nos enfrentamos a problemas que la limpieza básica no puede resolver, como:
- **Inconsistencias de formato**: "Nueva York" vs "NY".
- **Errores de digitación**: "Neltflix" en lugar de "Netflix".
- **Datos Fragmentados**: Información repartida en varios archivos CSV.

Este notebook aborda estas complejidades transformando datos brutos en un catálogo consolidado y listo para el análisis profesional.

In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("darkgrid")
DATA_PATH = Path('netflix_titles.csv')
ORIGINALS_PATH = Path('netflix_originals.csv')

## 2. Auditoría del Catálogo

Antes de cualquier acción, realizamos una auditoría para cuantificar la "suciedad" de los datos.

In [ ]:
df_netflix = pd.read_csv(DATA_PATH)

print(f"Dimensiones del Dataset: {df_netflix.shape}")
print("\nTipos de datos:\n", df_netflix.dtypes)
print("\nValores faltantes:\n", df_netflix.isnull().sum().sort_values(ascending=False))

## 3. Resolución de Datos Faltantes y Outliers

Implementamos una estrategia mixta: eliminamos donde la información es crítica y missing, e imputamos donde es posible.

In [ ]:
# 1. Imputación de Rating (Categoría) usando la Moda
common_rating = df_netflix['rating'].mode()[0]
df_netflix['rating'] = df_netflix['rating'].fillna(common_rating)

# 2. Manejo de Outliers en la duración de películas
df_movies = df_netflix[df_netflix['type'] == 'Movie'].copy()
df_movies['minutes'] = df_movies['duration'].str.extract(r'(\d+)').astype(float)

# Filtramos películas extremadamente cortas (<40 min) o largas (>200 min)
df_movies_filtered = df_movies[(df_movies['minutes'] > 40) & (df_movies['minutes'] < 200)]

print(f"Registros originales: {len(df_movies)} | Registros tras filtrar outliers: {len(df_movies_filtered)}")

## 4. Normalización de Texto

El texto es la fuente más común de inconsistencias. Aplicamos un proceso de estandarización.

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = text.lower() # Minúsculas
    text = text.strip() # Quitar espacios extremos
    text = re.sub(r'[^\w\s]', '', text) # Quitar puntuación
    return text

df_netflix['title_clean'] = df_netflix['title'].apply(clean_text)
df_netflix.drop_duplicates(['title_clean', 'type'], keep='last', inplace=True)

df_netflix[['title', 'title_clean']].head()

## 5. El Puente Pedagógico: Lógica Difusa (*Fuzzy Matching*)

### El Problema de la Identidad
En bases de datos reales, "New York" y "NY" representan la misma entidad, pero para Python son strings totalmente diferentes. El *Fuzzy Matching* calcula la distancia de Levenshtein (cuántos cambios de caracteres se necesitan para que un texto sea igual a otro).

#### Diagrama de Flujo de Limpieza Avanzada
```
  [ Dataset A ]      [ Dataset B ]
          |                  |
          v                  v
  [ Limpieza Básica ]  [ Limpieza Básica ]
  (Nulls, Outliers)    (Nulls, Outliers)
          |                  |
          v                  v
  [ Normalización ]    [ Normalización ]
  (Lower, Regex)       (Lower, Regex)
          |                  |
          +----------+-------+
                     |
                     v
             [ Fuzzy Matching ]  <-- Resolve "NY" vs "New York"
                     |
                     v
             [ Data Merging ]    <-- Outer Join
                     |
                     v
             [ Final Dataset ]
```

In [ ]:
try:
    from fuzzywuzzy import process, fuzz
except ImportError:
    print("Por favor, instala fuzzywuzzy y python-Levenshtein")

states_canonical = ['New York', 'California', 'Washington', 'Hawaii']
df_states = pd.DataFrame({'raw_state': ['NY', 'CA', 'Washington DC', 'Hawai']})

def get_best_match(query, choices):
    # Retorna el mejor match y el score de similitud
    return process.extractOne(query, choices, scorer=fuzz.token_sort_ratio)

df_states[['match', 'score']] = df_states['raw_state'].apply(lambda x: pd.Series(get_best_match(x, states_canonical)))
df_states

## 6. Consolidación de Fuentes (Merging)

Fusionamos el catálogo general con la lista de originales de Netflix.

In [ ]:
df_originals = pd.read_csv(ORIGINALS_PATH)
df_originals = df_originals.rename(columns={'titles': 'title', 'years': 'release_year'})
df_originals['release_year'] = df_originals['release_year'].astype(int)

# Merge externo para conservar todos los títulos
df_final = pd.merge(df_originals, df_netflix, on=['title', 'type', 'release_year'], how='outer')

# Identificar si es Original o Catálogo
df_final['original'] = df_final['original'].fillna('Catalog')

# Eliminar duplicados resultantes del merge
df_final.drop_duplicates(['title'], keep='first', inplace=True)

df_final[['original', 'type']].value_counts()

## 📝 Ejercicios

1. **Fuzzy Matching Real**: Aplica la función de `fuzzywuzzy` a la columna `country` del dataset original para normalizar nombres de países mal escritos.
2. **Limpieza de Cast**: La columna `cast` contiene una lista de actores separados por comas. Crea una nueva columna llamada `main_actor` que contenga solo el primer actor de la lista.
3. **Análisis de Consistencia**: Encuentra cuántos títulos cambiaron su categoría (`type`) entre el dataset de originales y el catálogo general.

## 📋 Resumen de Técnicas Avanzadas

| Técnica | Herramienta | Propósito |
|---|---|---|
| **Normalización** | `re.sub()`, `.lower()` | Eliminar ruido y estandarizar texto |
| **Fuzzy Matching** | `fuzzywuzzy` | Resolver sinonimia y errores tipográficos |
| **Merging** | `pd.merge(how='outer')` | Consolidar múltiples fuentes de verdad |
| **Deduplicación** | `.drop_duplicates()` | Asegurar la unicidad de la entidad principal |